In [15]:
import os
import pandas as pd
import requests
import io
import urllib3
import psycopg2
import psycopg2.extras
from dotenv import load_dotenv

# Disabilita gli avvisi a schermo per le connessioni non verificate (Bypass SSL)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [16]:
# ==========================================
# 1. CARICAMENTO CREDENZIALI DAL FILE .ENV
# ==========================================
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASSWORD")


In [19]:
# ==========================================
# 2. ESTRAZIONE E TRASFORMAZIONE DATI (ECB)
# ==========================================
print("Download dei dati storici dalla BCE...")
url = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-hist.zip"
response = requests.get(url, verify=False)

# Decompressione al volo dell'archivio zip in memoria
df = pd.read_csv(io.BytesIO(response.content), compression='zip')

# Conversione date e filtro dal 3 gennaio 2000 in poi
df['Date'] = pd.to_datetime(df['Date'])
df = df[df['Date'] >= '2000-01-03']

# Selezione delle valute richieste (USD, GBP, CHF) rispetto all'EUR
df = df[['Date', 'USD', 'GBP', 'CHF']]

# Unpivot: trasformiamo le colonne in righe (base_currency, quote_currency, fx_rate)
df_melted = df.melt(id_vars=['Date'], var_name='quote_currency', value_name='fx_rate')

# Rimuoviamo eventuali valori nulli causati da festività di mercato
df_melted = df_melted.dropna(subset=['fx_rate'])

# Aggiungiamo i campi standard richiesti dallo schema raw.fx_rates
df_melted['base_currency'] = 'EUR'
df_melted['rate_type'] = 'OFFICIAL'
df_melted['source_name'] = 'ECB'
df_melted = df_melted.rename(columns={'Date': 'rate_date'})

# Ordiniamo le colonne esattamente come sono sulla tabella del database
colonne_finali = ['rate_date', 'base_currency', 'quote_currency', 'fx_rate', 'rate_type', 'source_name']
df_melted = df_melted[colonne_finali]

# Convertiamo il dataframe in una lista di tuple per l'inserimento massivo (bulk insert)
dati_da_inserire = [tuple(x) for x in df_melted.to_numpy()]

# ==========================================
# 3. CARICAMENTO NEL DATABASE POSTGRESQL
# ==========================================
print(f"Inizio inserimento di {len(dati_da_inserire)} tassi di cambio in raw.fx_rates...")

try:
    # Connessione al DB utilizzando i parametri del .env
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASS
    )
    cursor = conn.cursor()

    # Query di inserimento
    # (Usiamo la sintassi %s che psycopg2 interpreterà in modo sicuro)
    insert_query = """
        INSERT INTO raw.fx_rates
        (rate_date, base_currency, quote_currency, fx_rate, rate_type, source_name)
        VALUES %s
    """

    # execute_values è la funzione di psycopg2 ottimizzata per caricare migliaia di righe in pochi decimi di secondo
    psycopg2.extras.execute_values(cursor, insert_query, dati_da_inserire)

    # Confermiamo la transazione
    conn.commit()
    print("Inserimento completato con successo!")

except Exception as e:
    print(f"Errore durante l'inserimento nel DB: {e}")
    if 'conn' in locals() and conn:
        conn.rollback() # Annulla in caso di errore
finally:
    if 'conn' in locals() and conn:
        cursor.close()
        conn.close()
        print("Connessione al database chiusa.")

Download dei dati storici dalla BCE...
Inizio inserimento di 20541 tassi di cambio in raw.fx_rates...
Inserimento completato con successo!
Connessione al database chiusa.
